# Ligamento · E1 — cabezas mixtas softmax+delta

Campaña principal de E1 (§6 del protocolo v1.0 congelado). Condiciones **C1** (softmax), **C2** (delta),
**C3** (mix22 = 2 softmax + 2 delta) y **C4** (mix31/mix13, exploratorias), **8 semillas**, tareas **T1**
(capacidad, 6 cargas) + **T2** (correctabilidad).

**Convergencia (decisión de Maxi 2026-07-22):** acc@1 en {L96, L128} cada 500 pasos; converge cuando la
mejora en la ventana de 500 es < 0.5 pts; bloques uniformes de +2500 con **tope duro 10 000**. Las 8 semillas
de cada condición bajo el **mismo régimen**.

**Campaña por sesiones.** En T4 la campaña completa no entra en una sesión de Colab free, así que está
fraccionada en unidades de ~40 min. Corré las celdas 1–7 una vez por sesión y después la celda 8 (la maestra);
cuando la sesión se corte, volvé a correr todo y retoma donde quedó.

**Uso:** `Entorno de ejecución → Cambiar tipo de entorno → GPU`, luego `Ejecutar todo`.

In [ ]:
# 1) Hardware. T4 es lo que da Colab free; A100/L4 requieren unidades de computo pagas.
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'SIN GPU: Entorno de ejecucion -> Cambiar tipo -> GPU'

In [ ]:
# 2) Dependencias
!pip -q install optax >/dev/null
import jax; print('jax', jax.__version__, '| devices:', jax.devices())
assert jax.devices()[0].platform == 'gpu', 'No hay GPU visible para JAX. Activa GPU en el entorno.'

In [ ]:
# 3) Codigo pre-registrado (repo publico)
![ -d telar-ligamento ] && (cd telar-ligamento && git pull -q) || git clone -q https://github.com/SperanzaMax/telar-ligamento
!ls telar-ligamento/experimentos/E1

In [ ]:
# 4) Drive: checkpoints + resultados sobreviven al corte de sesion
from google.colab import drive
drive.mount('/content/drive')
import os; RESULTS='/content/drive/MyDrive/ligamento_e1'; os.makedirs(RESULTS, exist_ok=True)
print('resultados ->', RESULTS)

## Gate de pre-registro

E1 **no se lanza** si el protocolo madre o el prereg de seguimiento congelado no coinciden con su hash.
El `--requiere PREREG_SEGUIMIENTO_v1.1` implementa la **condición 3** del plan de arranque: el prereg de
seguimiento actualizado tiene que estar congelado y pusheado antes de correr.

In [ ]:
# 5) Verificar anclas (bloquea si algo cambio o si falta el prereg v1.1 congelado)
!cd telar-ligamento && python experimentos/verificar_anclas.py --requiere PREREG_SEGUIMIENTO_v1.1

In [ ]:
# 6) (opcional, ~2 min) Costo real en ESTA GPU. Solo cronometra el paso: no toca metricas de tarea.
#    Correlo una vez por tipo de GPU; el planificador despues se auto-calibra con las corridas reales.
!cd telar-ligamento && python experimentos/E1/bench_costo.py 30

In [ ]:
# 7) (opcional) Aviso por Telegram al terminar cada unidad. El token va SOLO por env, nunca al repo.
import os
os.environ['TG_TOKEN'] = ''        # <- pegar token aqui si se quiere aviso; vacio = sin avisos
os.environ['TG_CHAT']  = '7985522502'
print('avisos Telegram:', 'ON' if os.environ['TG_TOKEN'] else 'OFF')

## Celda maestra — UNA SESIÓN de trabajo

Colab free corta a las ~4 h, así que la campaña está fraccionada en **unidades atómicas**: una semilla,
un bloque de 2500 pasos (~40 min en T4). Cada sesión ejecuta las unidades que entren en el presupuesto y
**corta limpia**, dejando checkpoint en Drive. La siguiente sesión retoma sola: **volvé a ejecutar esta
misma celda**, tantas veces como haga falta.

- `PRESUPUESTO_MIN=210` (3.5 h) deja media hora de margen antes del corte de Colab. Nunca arranca una unidad
  que no entre: si no entra, corta y la deja para la próxima sesión.
- `CONDS=delta,softmax,mix22` son las tres condiciones que dan **todos** los veredictos pre-registrados
  (PS-1..PS-5, P1.1/P1.2/P1.3). `mix31,mix13` (C4) son exploratorias — segunda tanda, después.
- El orden de ejecución no afecta los resultados: cada semilla es determinista e independiente, así que
  fraccionar la campaña no toca nada del pre-registro.

In [ ]:
# 8) UNA SESION. Re-ejecutar esta celda cuantas veces haga falta: retoma donde quedo.
%env RESULTS_DIR=/content/drive/MyDrive/ligamento_e1
%env N_SEEDS=8
%env CONDS=delta,softmax,mix22
%env PRESUPUESTO_MIN=210
%env MODO=sesion
!cd telar-ligamento/experimentos/E1 && python e1_runner.py

In [ ]:
# 8b) Ver avance SIN entrenar nada: que falta y cuantas sesiones quedan. Segundos, no usa GPU.
%env RESULTS_DIR=/content/drive/MyDrive/ligamento_e1
%env N_SEEDS=8
%env CONDS=delta,softmax,mix22
%env MODO=estado
!cd telar-ligamento/experimentos/E1 && python e1_runner.py

## Cierre — juntar todo y emitir el informe

Cuando la celda 8b diga **«CAMPAÑA COMPLETA»**, esta celda junta los JSON de todas las sesiones desde Drive
y emite `E1_informe.md` con los veredictos del prereg v1.1: PS-1 con doble tabla y regla de discordancia,
PS-2, PS-4 (i/ii/iii), PS-5 con control por paso de parada, y P1.1/P1.2/P1.3 del protocolo madre.

(El informe se emite solo al terminar la última unidad; esta celda sirve para regenerarlo cuando quieras.)

In [ ]:
# 9) Forzar el informe agregado (junta todas las sesiones)
%env RESULTS_DIR=/content/drive/MyDrive/ligamento_e1
%env N_SEEDS=8
%env CONDS=delta,softmax,mix22
%env MODO=informe
!cd telar-ligamento/experimentos/E1 && python e1_runner.py

In [ ]:
# 10) Detalle por semilla (util para ver el avance fino)
import glob, json, os
for f in sorted(glob.glob('/content/drive/MyDrive/ligamento_e1/e1_*.json')):
    if f.endswith('_propio.json'):
        continue
    d = json.load(open(f))
    print(f"{d['cond']:<8} s{d['seed']} @{d['steps']:>5} conv={str(d['converged']):<5} "
          f"L96={d['capacity']['96']['1']:.3f} L128={d['capacity']['128']['1']:.3f} "
          f"T2@32={d['T2']['32']:.3f}")
ck = glob.glob('/content/drive/MyDrive/ligamento_e1/*.ckpt')
print(f"\ncheckpoints presentes: {len(ck)}")